# 03 - Neo4j Ingestion

**Task 1, Fase 3:** Load triples ke Neo4j database.

**Pipeline:** `Triples JSON → Load Neo4j → Verify`

⚠️ **Pastikan Neo4j sudah running sebelum menjalankan notebook ini.**

In [1]:
# === Setup ===
import sys
import json
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

print(f'Project root: {PROJECT_ROOT}')
print(f'Neo4j URI: {NEO4J_URI}')
print(f'Neo4j database: {NEO4J_DATABASE}')

Project root: d:\TA\llm-driven-legal-kg-visualization
Neo4j URI: bolt://localhost:7687
Neo4j database: experiment-2


## Step 0: Configuration

Set **PROMPT_ID**, **stage**, dan **target database**.

- `STAGE`: file mana yang di-load (`triples` / `validated` / `deduped`)
- `NEO4J_DATABASE`: database tujuan (default dari `.env`)

In [ ]:
# === Configuration ===
DOCUMENT_ID = 'POJK_11_2022'
PROMPT_ID = 'PROMPT_3'

# Stage: 'triples' (raw), 'validated', atau 'deduped'
STAGE = 'deduped'

# Build input path
INPUT_PATH = f'data/{STAGE}/{DOCUMENT_ID}_{PROMPT_ID}_triples.json'

print(f'Document: {DOCUMENT_ID}')
print(f'Prompt: {PROMPT_ID}')
print(f'Stage: {STAGE}')
print(f'Input: {INPUT_PATH}')
print(f'Target DB: {NEO4J_DATABASE}')

if os.path.exists(INPUT_PATH):
    print(f'✅ File found')
else:
    print(f'❌ File NOT found! Run 02_llm_extraction.ipynb first.')

Document: UU_11_2008
Prompt: PROMPT_3
Stage: deduped
Input: data/deduped/UU_11_2008_PROMPT_3_triples.json
Target DB: experiment-2
✅ File found


## Step 1: Load Data & Connect to Neo4j

In [3]:
# Load triples
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    triples_data = json.load(f)

print(f'Nodes: {triples_data["total_nodes"]}')
print(f'Edges: {triples_data["total_edges"]}')

Nodes: 406
Edges: 653


In [4]:
from pipeline.load.neo4j_loader import Neo4jLoader

loader = Neo4jLoader(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, database=NEO4J_DATABASE)

Connected to Neo4j at bolt://localhost:7687 (database: experiment-2)


## Step 2: Clear DB & Create Constraints

In [ ]:
# ⚠️ Clear database (uncomment to execute)
# loader.clear_database()
# loader.create_constraints()
# loader.create_indexes()

Database cleared.
Constraints created.
  Vector index created.
  Full-text index created.
Indexes created.


## Step 3: Load Nodes & Edges

In [6]:
# Load nodes
print(f'Loading {len(triples_data["nodes"])} nodes...')
loader.load_nodes(triples_data['nodes'])

# Load edges
print(f'\nLoading {len(triples_data["edges"])} edges...')
loader.load_edges(triples_data['edges'])

Loading 406 nodes...


Loading nodes:   0%|          | 0/406 [00:00<?, ?it/s]

Loading nodes: 100%|██████████| 406/406 [00:19<00:00, 20.41it/s]



Loading 653 edges...


Loading edges: 100%|██████████| 653/653 [00:03<00:00, 205.30it/s]


## Step 4: Verify

In [7]:
# Get database stats
stats = loader.get_stats()

print(f'=== Neo4j Database Stats ===')
print(f'Database: {NEO4J_DATABASE}')
print(f'Stage: {STAGE} | Prompt: {PROMPT_ID}')
print(f'Total nodes: {stats["total_nodes"]}')
print(f'Total edges: {stats["total_edges"]}')
print(f'\nNode labels:')
for label, count in sorted(stats['node_labels'].items(), key=lambda x: -x[1]):
    print(f'  {label:20s}: {count}')
print(f'\nRelationship types:')
for rel, count in sorted(stats['relationship_types'].items(), key=lambda x: -x[1]):
    print(f'  {rel:20s}: {count}')

=== Neo4j Database Stats ===
Database: experiment-2
Stage: deduped | Prompt: PROMPT_3
Total nodes: 405
Total edges: 653

Node labels:
  PerbuatanHukum      : 133
  Ayat                : 105
  Pasal               : 54
  EntitasHukum        : 54
  KonsepHukum         : 26
  Sanksi              : 16
  Bab                 : 13
  Regulasi            : 2
  Bagian              : 2

Relationship types:
  BERLAKU_UNTUK       : 180
  MENGATUR            : 136
  MERUJUK             : 121
  MEMILIKI_AYAT       : 105
  MEMUAT              : 69
  MENDEFINISIKAN      : 23
  MENETAPKAN_SANKSI   : 19


In [8]:
# Test a simple Cypher query
with loader._session() as session:
    result = session.run("""
        MATCH (p:Pasal)-[r:MENGATUR]->(ph:PerbuatanHukum)
        RETURN p.label AS pasal, ph.label AS perbuatan
        LIMIT 10
    """)
    print('=== Sample: Pasal → MENGATUR → PerbuatanHukum ===')
    for r in result:
        print(f'  {r["pasal"]} → {r["perbuatan"]}')

=== Sample: Pasal → MENGATUR → PerbuatanHukum ===
  Pasal 2 → melakukan perbuatan hukum sebagaimana diatur dalam Undang-Undang ini
  Pasal 3 → Pemanfaatan Teknologi Informasi dan Transaksi Elektronik berdasarkan asas kepastian hukum, manfaat, kehati-hatian, iktikad baik, dan kebebasan memilih teknologi atau netral teknologi
  Pasal 4 → Pemanfaatan Teknologi Informasi dan Transaksi Elektronik dilaksanakan dengan tujuan
  Pasal 6 → dianggap sah sepanjang informasi yang tercantum di dalamnya dapat diakses, ditampilkan, dijamin keutuhannya, dan dapat dipertanggungjawabkan sehingga menerangkan suatu keadaan
  Pasal 7 → harus memastikan bahwa Informasi Elektronik dan/atau Dokumen Elektronik yang ada padanya berasal dari Sistem Elektronik yang memenuhi syarat berdasarkan Peraturan Perundang-undangan
  Pasal 9 → harus menyediakan informasi yang lengkap dan benar berkaitan dengan syarat kontrak, produsen, dan produk yang ditawarkan
  Pasal 14 → harus menyediakan informasi yang akurat, jelas, da

In [9]:
# Clean up
loader.close()
print('Neo4j connection closed.')

Neo4j connection closed.


## Summary

Knowledge Graph loaded ke Neo4j.

**Loaded:** `{STAGE}/{DOCUMENT_ID}_{PROMPT_ID}_triples.json` → database `{NEO4J_DATABASE}`

**Bisa diakses di:** http://localhost:7474 (Neo4j Browser)

**Next:** Run `04_evaluation.ipynb` to evaluate KG quality.